# CUDA: Matrix Multiplication
**Objective:** Implement matrix multiplication on GPU using CUDA C with shared memory optimization.

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Write CUDA matrix multiplication kernel
cuda_code = """
#include <stdio.h>
#include <cuda_runtime.h>
#include <math.h>

#define BLOCK_SIZE 16

// Naive matrix multiplication
__global__ void matMul_naive(float *A, float *B, float *C, int n) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    
    if (row < n && col < n) {
        float sum = 0.0f;
        for (int k = 0; k < n; k++) {
            sum += A[row * n + k] * B[k * n + col];
        }
        C[row * n + col] = sum;
    }
}

// Optimized using shared memory
__global__ void matMul_shared(float *A, float *B, float *C, int n) {
    __shared__ float sA[BLOCK_SIZE][BLOCK_SIZE];
    __shared__ float sB[BLOCK_SIZE][BLOCK_SIZE];
    
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    float sum = 0.0f;
    
    for (int k = 0; k < n; k += BLOCK_SIZE) {
        sA[threadIdx.y][threadIdx.x] = A[row * n + k + threadIdx.x];
        sB[threadIdx.y][threadIdx.x] = B[(k + threadIdx.y) * n + col];
        __syncthreads();
        
        for (int i = 0; i < BLOCK_SIZE; i++) {
            sum += sA[threadIdx.y][i] * sB[i][threadIdx.x];
        }
        __syncthreads();
    }
    
    if (row < n && col < n) {
        C[row * n + col] = sum;
    }
}

int main() {
    int n = 1024;
    size_t bytes = n * n * sizeof(float);
    
    // Host arrays
    float *h_A = (float*)malloc(bytes);
    float *h_B = (float*)malloc(bytes);
    float *h_C_naive = (float*)malloc(bytes);
    float *h_C_shared = (float*)malloc(bytes);
    
    // Initialize
    for (int i = 0; i < n * n; i++) {
        h_A[i] = 1.0f;
        h_B[i] = 1.0f;
    }
    
    // Device arrays
    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, bytes);
    cudaMalloc(&d_B, bytes);
    cudaMalloc(&d_C, bytes);
    
    // Copy to device
    cudaMemcpy(d_A, h_A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, bytes, cudaMemcpyHostToDevice);
    
    dim3 blockDim(BLOCK_SIZE, BLOCK_SIZE);
    dim3 gridDim((n + BLOCK_SIZE - 1) / BLOCK_SIZE, (n + BLOCK_SIZE - 1) / BLOCK_SIZE);
    
    // Naive kernel
    cudaEvent_t start, stop;
    cudaEventCreate(&start); cudaEventCreate(&stop);
    
    cudaEventRecord(start);
    matMul_naive<<<gridDim, blockDim>>>(d_A, d_B, d_C, n);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    float time_naive;
    cudaEventElapsedTime(&time_naive, start, stop);
    
    cudaMemcpy(h_C_naive, d_C, bytes, cudaMemcpyDeviceToHost);
    
    // Shared memory kernel
    cudaEventRecord(start);
    matMul_shared<<<gridDim, blockDim>>>(d_A, d_B, d_C, n);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    float time_shared;
    cudaEventElapsedTime(&time_shared, start, stop);
    
    cudaMemcpy(h_C_shared, d_C, bytes, cudaMemcpyDeviceToHost);
    
    printf("Matrix size: %d x %d\\n", n, n);
    printf("Naive time: %.3f ms\\n", time_naive);
    printf("Shared memory time: %.3f ms\\n", time_shared);
    printf("Speedup: %.2fx\\n", time_naive / time_shared);
    printf("Sample result C[0][0]: %.1f\\n", h_C_shared[0]);
    
    // Cleanup
    free(h_A); free(h_B); free(h_C_naive); free(h_C_shared);
    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    cudaEventDestroy(start); cudaEventDestroy(stop);
    
    return 0;
}
"""

with open('matrix_mul.cu', 'w') as f:
    f.write(cuda_code)
print('CUDA code saved')

In [ ]:
# Compile
!nvcc -o matrix_mul matrix_mul.cu

In [ ]:
# Run
!./matrix_mul